In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# read data
import sys
sys.path.append('../..')
import data_preprocessing.coil20 as coil20
from importlib import reload
reload(coil20)

# training process
import algorithm.algo1 as algo1
reload(algo1)

# evaluation
import algorithm.kmeans as kmeans
import eval_metrics.ac

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Using device]', device)

[Using device] cuda


In [ ]:
import argparse

parser = argparse.ArgumentParser(description='Train a model for clustering')
parser.add_argument('--n_clusters', type=int, default=20, help='number of clusters')
parser.add_argument('--loss', type=str, default='reconstruction_loss_without_regularization', help='loss function')
parser.add_argument('--lr', type=float, default=1e-4, help='learning rate')
parser.add_argument('--lamda', type=float, default=1, help='regularization parameter')
args = parser.parse_args()

CLUSTER_NUM = args.n_clusters
LOSS_FUNCTION_NAME = args.loss
LOSS_FUNCTION = getattr(origin, LOSS_FUNCTION_NAME)
LEARNING_RATE = args.lr
if LOSS_FUNCTION_NAME == 'reconstruction_loss_without_regularization':
    LAMBDA = None
else:
    LAMBDA = args.lamda

In [4]:
CLUSTER_NUM = 8
LAMBDA = 8

# k = 10, lambda=10 (越大越不易nan); k-means多几次可达79%

In [5]:
# read data
mat, label = coil20.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, 
                              random_choose_class=True)
mat = torch.tensor(mat, dtype=torch.float32)
mat -= torch.mean(mat)

print(f'[Using classes] {np.unique(np.array(label))}')

# compute the similarity matrix using cosine similarity
mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
A = mat_normalized @ mat_normalized.T
A.to(device)
pass

[Using classes] [ 2  4  8  9 10 12 16 20]


In [6]:
reload(algo1)
H = torch.randn(A.shape[0], CLUSTER_NUM) + 1
H.to(device)

H_out, P_out = algo1.algo1(A=A, H=H, 
                           k=CLUSTER_NUM, lamda=LAMBDA, 
                           max_iter=10000000,
                           tol=1e-15, print_every=10)

iter 10 done, update = 0.012865625321865082
iter 20 done, update = 0.004235208034515381
iter 30 done, update = 0.0010501816868782043
iter 40 done, update = 0.0007146820425987244
iter 50 done, update = 0.0004208311438560486
iter 60 done, update = 0.00015315040946006775
iter 70 done, update = 8.60174186527729e-05
iter 80 done, update = 6.541935727000237e-05
iter 90 done, update = 6.928248330950737e-05
iter 100 done, update = 6.32372684776783e-05
iter 110 done, update = 2.0733103156089783e-05
iter 120 done, update = 7.670372724533081e-06
iter 130 done, update = 2.893153578042984e-06
iter 140 done, update = 3.3890828490257263e-06
iter 150 done, update = 6.757676601409912e-06
iter 160 done, update = 4.773493856191635e-06
iter 170 done, update = 9.051058441400528e-06
iter 180 done, update = 1.4252960681915283e-05
iter 190 done, update = 1.953495666384697e-05
iter 200 done, update = 3.535649739205837e-05
iter 210 done, update = 4.723528400063515e-05
iter 220 done, update = 9.081559255719185e-

In [7]:
torch.norm(H_out - P_out)

tensor(0.0156)

In [ ]:
pred_labels, centroids = kmeans.kmeans(H_out.detach().cpu().numpy(), CLUSTER_NUM)
acc = eval_metrics.ac.ac(label, pred_labels)
print(f'[Accuracy] {acc}')